In [2]:
%load_ext autoreload
%autoreload 2
import numpy as np
import D4CMPP_TLC
import SSC
from rdkit import Chem
import matplotlib.pyplot as plt

In [8]:
CLI_segment = SSC.src.SSCAnalyzer.SSCAnalyzer_v1p3("CLI_segment_attention_simple_C_paper_Rf_620_20260627_084202")
BLI_segment = SSC.src.SSCAnalyzer.SSCAnalyzer_v1p3("BLI_segment_attention_simple_C_paper_Rf_620_20260627_084209")


founded model: ./_Models/CLI_segment_attention_simple_C_paper_Rf_620_20260627_084202
Device: cuda:3
#params: 204131
Loading params from ./_Models/CLI_segment_attention_simple_C_paper_Rf_620_20260627_084202/final.pth
Sculptor Index: (6, 2, 0)
founded model: ./_Models/BLI_segment_attention_simple_C_paper_Rf_620_20260627_084209
Device: cuda:2
#params: 204131
Loading params from ./_Models/BLI_segment_attention_simple_C_paper_Rf_620_20260627_084209/final.pth
Sculptor Index: (6, 2, 0)


In [9]:
CLI_segment.get_score("Brc1cccc(N2CCCCC2)c1",
          "CCCCCC",
          "O=C(OCC)C",
          0.8,
          0.2,)

Positional arguments are provided. Note that arguments should be in the order of ['Smiles', 'eluent1_smiles', 'eluent2_smiles'] and ['ratio1', 'ratio2'].


{'y': array([[0.60966367]], dtype=float32),
 'attn1': array([[-0.01140025],
        [-0.0333301 ],
        [-0.02975424]], dtype=float32),
 'attn2': array([[ 0.05517191],
        [-0.07224819],
        [ 0.00269289]], dtype=float32),
 'attn1_ratio': array([[-0.0091202 ],
        [-0.02666408],
        [-0.02380339]], dtype=float32),
 'attn2_ratio': array([[ 0.01103438],
        [-0.01444964],
        [ 0.00053858]], dtype=float32),
 'attn_size': array([[3]])}

In [5]:
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import HTML, display
import html
import numpy as np


def _as_1d_score(score_value):
    return np.asarray(score_value, dtype=float).reshape(-1)


def _fragment_atom_indices(fragment):
    return [int(a) for a in fragment.atoms]


def _subgroup_scores_to_atom_scores(mol, fragments, subgroup_scores):
    atom_values = {i: [] for i in range(mol.GetNumAtoms())}

    for fragment, score_value in zip(fragments, subgroup_scores):
        for atom_idx in _fragment_atom_indices(fragment):
            atom_values[atom_idx].append(float(score_value))

    return {
        atom_idx: float(np.mean(values))
        for atom_idx, values in atom_values.items()
        if values
    }


def _blue_white_red(score_value, vmin, vmax):
    if np.isclose(vmin, vmax):
        return (0.95, 0.95, 0.95)

    t = float((score_value - vmin) / (vmax - vmin))
    t = min(max(t, 0.0), 1.0)

    if t < 0.5:
        # blue -> white
        u = t / 0.5
        return (u, u, 1.0)

    # white -> red
    u = (t - 0.5) / 0.5
    return (1.0, 1.0 - u, 1.0 - u)


def _draw_attention_svg(smiles, fragments, subgroup_scores, title, vmin, vmax, width=360, height=280):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid molecule SMILES: {smiles}")

    mol = Chem.Mol(mol)
    atom_scores = _subgroup_scores_to_atom_scores(mol, fragments, subgroup_scores)
    highlight_atoms = list(atom_scores.keys())
    highlight_colors = {
        atom_idx: _blue_white_red(atom_scores[atom_idx], vmin, vmax)
        for atom_idx in highlight_atoms
    }
    highlight_radii = {
        atom_idx: 0.32 + 0.28 * abs((atom_scores[atom_idx] - vmin) / (vmax - vmin))
        if not np.isclose(vmin, vmax) else 0.35
        for atom_idx in highlight_atoms
    }

    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addAtomIndices = True
    drawer.DrawMolecule(
        mol,
        legend=title,
        highlightAtoms=highlight_atoms,
        highlightAtomColors=highlight_colors,
        highlightAtomRadii=highlight_radii,
    )
    drawer.FinishDrawing()
    return drawer.GetDrawingText().replace("svg:", "")


def _score_table_html(fragments, subgroup_scores, score_key, eluent_smiles=None):
    rows = []
    if eluent_smiles is not None:
        rows.append(
            f"<div style='margin: 6px 0 8px 0;'><b>eluent smiles</b>: "
            f"<code>{html.escape(str(eluent_smiles))}</code></div>"
        )

    rows.append("<table style='border-collapse: collapse; font-size: 12px;'>")
    rows.append(
        "<tr>"
        "<th style='text-align:right; padding:2px 8px;'>frag</th>"
        "<th style='text-align:left; padding:2px 8px;'>frag_smiles</th>"
        "<th style='text-align:left; padding:2px 8px;'>atoms</th>"
        f"<th style='text-align:right; padding:2px 8px;'>{html.escape(score_key)}</th>"
        "</tr>"
    )
    for idx, (fragment, score_value) in enumerate(zip(fragments, subgroup_scores)):
        rows.append(
            "<tr>"
            f"<td style='text-align:right; padding:2px 8px;'>{idx}</td>"
            f"<td style='padding:2px 8px;'><code>{html.escape(str(fragment.smiles))}</code></td>"
            f"<td style='padding:2px 8px;'>{_fragment_atom_indices(fragment)}</td>"
            f"<td style='text-align:right; padding:2px 8px;'>{float(score_value):.6g}</td>"
            "</tr>"
        )
    rows.append("</table>")
    return "\n".join(rows)


def _attention_keys(score, ratio=False):
    if "attn1" in score and "attn2" in score:
        keys = ("attn1_ratio", "attn2_ratio") if ratio else ("attn1", "attn2")
        missing = [key for key in keys if key not in score]
        if missing:
            raise KeyError(f"Missing score key(s): {missing}. Available keys: {list(score.keys())}")
        return list(keys)

    if ratio and "attn_ratio" in score:
        return ["attn_ratio"]
    if "attn" in score:
        # Some BLI segment models already fold the eluent ratio into a single attn value.
        return ["attn"]

    expected = "attn_ratio" if ratio else "attn"
    raise KeyError(f"Missing score key: {expected}. Available keys: {list(score.keys())}")


def visualize_segment_attention(
    analyzer,
    smiles,
    eluent1_smiles,
    eluent2_smiles,
    ratio1,
    ratio2,
    *,
    ratio=False,
    score=None,
    width=360,
    height=280,
):
    """Visualize subgroup attention returned by analyzer.get_score().

    ratio=False uses attn or attn1/attn2.
    ratio=True uses attn_ratio or attn1_ratio/attn2_ratio when present.
    Single-attn BLI outputs may only have attn because ratio is already part of the model input.
    """
    if score is None:
        score = analyzer.get_score(smiles, eluent1_smiles, eluent2_smiles, ratio1, ratio2)

    fragments = analyzer.get_fragment(smiles)
    keys = _attention_keys(score, ratio=ratio)
    score_arrays = [_as_1d_score(score[key]) for key in keys]

    expected_size = len(fragments)
    for key, values in zip(keys, score_arrays):
        if len(values) != expected_size:
            raise ValueError(
                f"{key} length ({len(values)}) does not match fragment count ({expected_size})."
            )

    all_values = np.concatenate(score_arrays)
    vmin = float(np.min(all_values))
    vmax = float(np.max(all_values))

    combined_eluent_label = (
        f"{eluent1_smiles} ({ratio1:g}) + {eluent2_smiles} ({ratio2:g})"
    )
    eluent_by_key = {
        "attn": combined_eluent_label,
        "attn_ratio": combined_eluent_label,
        "attn1": eluent1_smiles,
        "attn1_ratio": eluent1_smiles,
        "attn2": eluent2_smiles,
        "attn2_ratio": eluent2_smiles,
    }

    panels = []
    for key, values in zip(keys, score_arrays):
        eluent_smiles = eluent_by_key.get(key)
        title = key if eluent_smiles is None else f"{key}: {eluent_smiles}"
        svg = _draw_attention_svg(
            smiles,
            fragments,
            values,
            title=title,
            vmin=vmin,
            vmax=vmax,
            width=width,
            height=height,
        )
        table = _score_table_html(fragments, values, key, eluent_smiles=eluent_smiles)
        panels.append(
            "<div style='display:inline-block; vertical-align:top; margin:0 18px 18px 0;'>"
            f"{svg}"
            f"{table}"
            "</div>"
        )

    caption = (
        "<div style='margin-bottom:8px; font-size:13px;'>"
        "Color scale: low attention = blue, high attention = red. "
        f"Range: {vmin:.6g} to {vmax:.6g}."
        "</div>"
    )
    display(HTML(caption + "<div>" + "\n".join(panels) + "</div>"))
    return score


# Example:
# score = visualize_segment_attention(
#     a_eq,
#     "Brc1cccc(N2CCCCC2)c1",
#     "CCCCCC",
#     "O=C(OCC)C",
#     0.8,
#     0.2,
#     ratio=False,
# )

In [10]:
score = visualize_segment_attention(
    CLI_segment,
    "Brc1cccc(N2CCCCC2)c1",
    "CCCCCC",
    "O=C(OCC)C",
    0.8,
    0.2,
    ratio=True,
)

Positional arguments are provided. Note that arguments should be in the order of ['Smiles', 'eluent1_smiles', 'eluent2_smiles'] and ['ratio1', 'ratio2'].


frag,frag_smiles,atoms,attn1_ratio
0,*Br,[0],-0.0091202
1,C1CCNCC1,"[6, 7, 8, 9, 10, 11]",-0.0266641
2,c1ccccc1,"[1, 2, 3, 4, 5, 12]",-0.0238034
frag,frag_smiles,atoms,attn2_ratio
0,*Br,[0],0.0110344
1,C1CCNCC1,"[6, 7, 8, 9, 10, 11]",-0.0144496
2,c1ccccc1,"[1, 2, 3, 4, 5, 12]",0.000538577


In [11]:
score = visualize_segment_attention(
    BLI_segment,
    "Brc1cccc(N2CCCCC2)c1",
    "CCCCCC",
    "O=C(OCC)C",
    0.8,
    0.2,
    ratio=True,
)

Positional arguments are provided. Note that arguments should be in the order of ['Smiles', 'eluent1_smiles', 'eluent2_smiles'] and ['ratio1', 'ratio2'].


KeyError: "Missing score key: attn_ratio. Available keys: ['y', 'attn', 'attn_size']"

In [33]:
print(a.scaler.scaler.mean_, a.scaler.scaler.var_)

[0.44897715] [0.0452323]
